In [ ]:
%load_ext autoreload
%autoreload 2

import edunn as nn
import numpy as np


# Cross-Entropy Layer

In this exercise you must implement the `CrossEntropyWithLabels` error layer, which lets you compute the error of a model that outputs probabilities in terms of distances between distributions. 

In this case, `WithLabels` indicates that the true probability distribution (obtained from the dataset) is actually encoded with labels, so that for a problem with `C=3` classes, if an example belongs to class 2 (counting from 0), then its label is `2`. This is a convenient way of specifying that its encoding as a probability distribution would be `[0,0,1]`, that is, a vector of `3` elements where element `2` (again, counting from 0) has probability 1 and the rest 0.



# Forward method


The `forward` method of the `CrossEntropyWithLabels` layer assumes that its input `y` is a probability distribution, that is, `C` positive values that sum to 1, where `C` is the number of classes. Likewise, `y_true` is a label indicating which of the `C` classes is the correct one.

For example, if $y=(0.3,0.4,0.3)$ and $y_{true}=2$ there will be a considerable error, since the value $y_{true}=2$ indicated that the expected distribution was  $y=(0,0,1)$. So the values $0.3$ and $0.4$ of classes 0 and 1 should go down, and the value $0.3$ of class 2 should go up

Cross-entropy quantifies this error by computing the negative logarithm of the probability of the correct 
class ($-ln(y_{y_{true}})$), in this case, of class 2 ($-ln(y_2)$). So, 

$$CrossEntropy(y,y_true)=CrossEntropy((0.3,0.4,0.3),2)=-ln(0.3)=1.20$$

Once again, in this case the value $0.3$ was chosen because it is the one sitting at index 2 of the vector $y$; in other words, another way of writing the above would be:

$$E(y,y_{true})=-ln(y_{y_{true}})=-ln(0.3)=1.20$$

The reason we use the function $-ln(0.3)$ to penalize is that if the probability of the correct class is 1, then 

$$-ln(y_{y_{true}})=-ln(1)=-0=0$$

and  there is no penalty. Otherwise, the output of $-ln$ will be positive and will signal an error. In this way we penalize the probability of the correct class for failing to reach 1. We can easily visualize this in a plot of the function $-ln(x)$:

<img src="img/cross_entropy.png" width="400">

Finally, since the values of $y$ are normalized, there is no need to penalize the remaining probabilities for being greater than 0; if the error drives the probability of the correct class to 1, then the rest will necessarily be 0. For this reason (among others), cross-entropy is a good match for the softmax function when training classification models.

In the case of a batch of examples, the computation is independent for each example.

Implement the `forward` method of the `CrossEntropyWithLabels` class:


In [ ]:
y = np.array([[1,0],
             [0.5,0.5],
              [0.5,0.5],
             ])
y_true = np.array([0,0,1])


layer=nn.CrossEntropyWithLabels()
E=-np.log(np.array([[1],[0.5],[0.5]]))

nn.utils.check_same(E,layer.forward(y_true,y))

# Backward method

Since deriving the equations for the `backward` method of the cross-entropy is a bit lengthy, we leave you [this note](http://facundoq.github.io/edunn/material/crossentropy_derivative) with the derivation of every case. 

Again, since this error is computed per example, the calculations are independent for each row.

Implement the `backward` method of the `CrossEntropyWithLabels` class:

In [ ]:
# number of random values of x and δEδy to generate and test gradients
samples = 100
batch_size=2
features_in=3
features_out=5
input_shape=(batch_size,features_in)


layer=nn.CrossEntropyWithLabels()
nn.utils.check_gradient.cross_entropy_labels(layer,input_shape,samples=samples,tolerance=1e-5)    


# Logistic Regression applied to Flower Classification

Now that we have all the pieces, we can define and train our first logistic regression model to classify the flowers of the [Iris](https://www.kaggle.com/uciml/iris) dataset.

This time we can finally do it with cross-entropy; while the results here are similar in terms of accuracy, the model now has a convex error, which makes the optimization easier.


In [ ]:

# Load the data with the outputs as labels
# (note: class labels start at 0)
x,y,classes=nn.datasets.load_classification("iris")
# data normalization
x = (x-x.mean(axis=0))/x.std(axis=0)
n, din = x.shape
# compute the number of classes
classes = y.max()+1
print("Sizes of x and y:", x.shape,y.shape)

#Logistic regression model, 
# it has `din` input dimensions (4 for iris)
# and `classes` outputs `3 for iris`
model = nn.LogisticRegression(din,classes)
# Mean squared error
error = nn.MeanError(nn.CrossEntropyWithLabels())
optimizer = nn.GradientDescent(lr=0.1,epochs=1000,batch_size=32)

# Optimization algorithm
history = optimizer.optimize(model,x,y,error)
nn.plot.plot_history(history,error_name=error.name)


print("Model metrics:")
y_pred=model.forward(x)
y_pred_labels=nn.utils.onehot2labels(y_pred)
nn.metrics.classification_summary(y,y_pred_labels)